# 📊 ĐỒ ÁN PHÂN KHÚC KHÁCH HÀNG BÁN BUÔN (WHOLESALE CUSTOMERS CLUSTERING)
## 🚀 Thuật toán K-Means | K-Means++ | Mini-Batch K-Means (Pure NumPy - Không dùng scikit-learn)

**Tác giả:** ntdat2036  
**Repository:** [kmean_k-_minipatch](https://github.com/ntdat2036/kmean_k-_minipatch)  

---
### 🎯 Mục tiêu đồ án:
1. Tự cài đặt 100% các thuật toán phân cụm **K-Means**, **K-Means++**, **MiniBatchKMeans**, **StandardScaler**, **PCA**, và các độ đo đánh giá (**Silhouette**, **Calinski-Harabasz**, **Davies-Bouldin**) bằng **NumPy thuần**.
2. Áp dụng quy trình Tiền xử lý dữ liệu tập trung (`preprocess.py`): Biến đổi log1p & tạo các tỷ lệ chi tiêu ngành hàng.
3. Tìm số cụm $K$ tối ưu bằng phương pháp Elbow & Silhouette Score.
4. Đánh giá và so sánh hiệu năng 3 thuật toán phân cụm trên cùng bộ dữ liệu Wholesale Customers.
5. Giảm chiều dữ liệu bằng PCA để trực quan hóa cụm trên mặt phẳng 2D.
6. Lập hồ sơ kinh doanh động cho từng phân khúc (VIP / HoReCa / Retail) và suy luận trên dữ liệu khách hàng mới.

## 1. Import Thư viện & Module Tự Viết

In [ ]:
import sys
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import module tự viết thuần NumPy
from model import (
    StandardScaler,
    KMeans,
    KMeansPlusPlus,
    MiniBatchKMeans,
    PCA,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    Pipeline,
    map_cluster_profiles
)
from preprocess import validate_input_data, build_features, FEATURE_COLS

print("✅ Import thành công các module từ model.py và preprocess.py")

## 2. Nạp Dữ Liệu & Tiền Xử Lý (Feature Engineering)

In [ ]:
data_path = "Wholesale customers data.csv"
df_raw = pd.read_csv(data_path)
print(f"📊 Số mẫu dữ liệu: {df_raw.shape[0]}, Số cột ban đầu: {df_raw.shape[1]}")
display(df_raw.head())

# Kiểm tra hợp lệ & Trích xuất đặc trưng tập trung
X_processed = build_features(df_raw)
print(f"✨ Số đặc trưng sau Feature Engineering: {X_processed.shape[1]}")
print("Danh sách các đặc trưng:", list(X_processed.columns))

# Chuẩn hóa Z-score
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed.values)
print("✅ Đã chuẩn hóa Z-score dữ liệu.")

## 3. Phân Tích Chọn Số Cụm K Tối Ưu (Elbow & Silhouette)

In [ ]:
k_values = list(range(2, 9))
inertias = []
sil_scores = []

for k in k_values:
    km_eval = KMeansPlusPlus(n_clusters=k, max_iter=300, n_init=5, random_state=42)
    labels_eval = km_eval.fit_predict(X_scaled)
    inertias.append(km_eval.inertia_)
    sil = silhouette_score(X_scaled, labels_eval)
    sil_scores.append(sil)

# Vẽ đồ thị Elbow & Silhouette Score
fig, ax1 = plt.subplots(figsize=(10, 5))
color = "tab:red"
ax1.set_xlabel("Số cụm (K)", fontsize=11)
ax1.set_ylabel("Inertia (WCSS)", color=color, fontsize=11)
ax1.plot(k_values, inertias, "o--", color=color, linewidth=2, label="Inertia")
ax1.tick_params(axis="y", labelcolor=color)
ax1.grid(True, linestyle="--", alpha=0.5)

ax2 = ax1.twinx()
color = "tab:blue"
ax2.set_ylabel("Silhouette Score", color=color, fontsize=11)
ax2.plot(k_values, sil_scores, "s-", color=color, linewidth=2, label="Silhouette Score")
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Phương pháp Elbow & Silhouette Score chọn K tối ưu", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

best_k_idx = int(np.argmax(sil_scores))
best_k = k_values[best_k_idx]
print(f"💡 Số cụm K đạt Silhouette cao nhất: K = {best_k} (Silhouette = {sil_scores[best_k_idx]:.4f})")
print("📌 Lưu ý nghiệp vụ: Mặc dù K = 2 đạt Silhouette cao nhất về mặt toán học, mô hình giữ K = 3 vì khớp hoàn hảo với 3 phân khúc thực tế (VIP, HoReCa, Retail).")

## 4. Huấn Luyện & So Sánh 3 Thuật Toán Phân Cụm (K = 3)

In [ ]:
K_CHOICE = 3
N_INIT = 10
SEED = 42

models_dict = {
    "KMeans (Random Init)": KMeans(n_clusters=K_CHOICE, n_init=N_INIT, random_state=SEED),
    "KMeans++ (D^2+Oversampling)": KMeansPlusPlus(n_clusters=K_CHOICE, n_init=N_INIT, random_state=SEED),
    "MiniBatchKMeans (Online)": MiniBatchKMeans(n_clusters=K_CHOICE, n_init=N_INIT, batch_size=100, random_state=SEED)
}

results = {}
for name, model in models_dict.items():
    t0 = time.time()
    labels = model.fit_predict(X_scaled)
    t_elapsed = time.time() - t0
    
    sil = silhouette_score(X_scaled, labels)
    chi = calinski_harabasz_score(X_scaled, labels)
    dbi = davies_bouldin_score(X_scaled, labels)
    
    results[name] = {
        "model": model,
        "labels": labels,
        "time": t_elapsed,
        "inertia": model.inertia_,
        "silhouette": sil,
        "chi": chi,
        "dbi": dbi
    }

df_summary = pd.DataFrame([
    {
        "Thuật toán": name,
        "Thời gian (s)": round(res["time"], 4),
        "Inertia (WCSS)": round(res["inertia"], 2),
        "Silhouette Score": round(res["silhouette"], 4),
        "Calinski-Harabasz": round(res["chi"], 2),
        "Davies-Bouldin": round(res["dbi"], 4)
    }
    for name, res in results.items()
])
display(df_summary)

## 5. Trực Quan Hóa PCA & Đường Cong Hội Tụ

In [ ]:
# Giảm chiều xuống 2D bằng PCA tự viết
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f"Trực quan hóa PCA 2D — Tỷ lệ phương sai giải thích: {pca.explained_variance_ratio_}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
CLUSTER_COLORS = ["#E74C3C", "#2ECC71", "#3498DB"]

for idx, (name, res) in enumerate(results.items()):
    ax = axes[idx]
    labels = res["labels"]
    for k in range(K_CHOICE):
        mask = (labels == k)
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f"Cụm {k}", color=CLUSTER_COLORS[k], alpha=0.7)
    ax.set_title(f"{name}\nSilhouette: {res['silhouette']:.4f}", fontsize=11, fontweight="bold")
    ax.set_xlabel("PCA Component 1")
    ax.set_ylabel("PCA Component 2")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Lập Hồ Sơ Phân Cụm Động (Dynamic Profiling) & Lưu Mô Hình

In [ ]:
# Chọn mô hình tốt nhất (MiniBatchKMeans)
best_key = "MiniBatchKMeans (Online)"
best_model = results[best_key]["model"]
best_labels = results[best_key]["labels"]

# Lập hồ sơ cụm
profiles = map_cluster_profiles(df_raw, best_labels)
print("📋 Hồ sơ nhãn kinh doanh các cụm:")
for cid, desc in profiles.items():
    print(f"  - Cụm {cid}: {desc}")

# Lưu hồ sơ cụm ra JSON
profiles_to_save = {str(k): v for k, v in profiles.items()}
with open("cluster_profiles.json", "w", encoding="utf-8") as f:
    json.dump(profiles_to_save, f, ensure_ascii=False, indent=2)
print("💾 Đã lưu cluster_profiles.json")

# Đóng gói và lưu Pipeline
pipeline = Pipeline([("scaler", scaler), ("kmeans", best_model)])
pipeline.save("kmeans_pipeline.pkl")

## 7. Suy Luận Trên Dữ Liệu Khách Hàng Mới (Inference)

In [ ]:
# Nạp lại pipeline và profiles
loaded_pipeline = Pipeline.load("kmeans_pipeline.pkl")
with open("cluster_profiles.json", "r", encoding="utf-8") as f:
    loaded_profiles = {int(k): v for k, v in json.load(f).items()}

# Dữ liệu khách hàng mới thử nghiệm
sample_customers = pd.DataFrame([
    {"Fresh": 35000, "Milk": 2000, "Grocery": 3000, "Frozen": 9000, "Detergents_Paper": 400, "Delicassen": 1500},
    {"Fresh": 2000, "Milk": 12000, "Grocery": 18000, "Frozen": 800, "Detergents_Paper": 7000, "Delicassen": 1200},
    {"Fresh": 55000, "Milk": 30000, "Grocery": 45000, "Frozen": 15000, "Detergents_Paper": 18000, "Delicassen": 10000}
])

X_new = build_features(sample_customers)
preds = loaded_pipeline.predict(X_new)
sample_customers["Cluster"] = preds

print("=" * 70)
print("  KẾT QUẢ PHÂN KHÚC KHÁCH HÀNG MỚI (INFERENCE)")
print("=" * 70)
for idx, row in sample_customers.iterrows():
    c_id = int(row["Cluster"])
    desc = loaded_profiles.get(c_id, f"Cụm {c_id}")
    print(f"  Khách hàng {idx + 1:2d} | Cụm {c_id} => {desc}")
print("=" * 70)